# 11 — Paper 2: Adversarial Training pada Pipeline SFM + Few-Shot (XGBoost)

**Dijalankan di SageMaker.** Ini notebook inti Paper 2. Paper 1 menunjukkan SFM + few-shot
menutup celah generalisasi lintas-jaringan (dimensi *distribution shift*). Paper 2 menambah
dimensi kedua: **ketahanan terhadap evasion adversarial**, dan bertanya:

> Dapatkah satu XGBoost pada 9 fitur SFM sekaligus (a) *generalisasi lintas-jaringan* via
> few-shot DAN (b) *tahan evasion* via adversarial training — tanpa saling mengorbankan?

**Empat varian model dilatih (2 arah: CIC→UNSW, UNSW→CIC):**
1. `baseline`        — source-only, clean (pembanding Paper 1).
2. `fewshot`         — source + 1% label target (generalisasi; Paper 1).
3. `adv`             — source + adversarial training (evasion-robust; Paper 1 T5).
4. `fewshot_adv`     — source + 1% target + adversarial training (**usulan Paper 2**).

Notebook ini **melatih + menyimpan** keempat model & scaler ke `paper2_models/` dan
meng-upload ke S3 `unsw-far/paper2/`. Evaluasi lengkap (clean/evasion/adaptive) di notebook 12.

> Building block (saliency finite-diff, FGSM, functional-preserving) konsisten dengan
> notebook 04/07/08.

In [ ]:
import importlib, subprocess, sys
for pkg,imp in [('pandas','pandas'),('numpy','numpy'),('scikit-learn','sklearn'),('xgboost','xgboost'),('boto3','boto3')]:
    try: importlib.import_module(imp)
    except ImportError: subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])
import pickle, os, json, time
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import matthews_corrcoef, f1_score, accuracy_score, confusion_matrix
from xgboost import XGBClassifier

CIC_PKL='../../CICDDoS2018/data/cleaned_100.pkl'
UNSW_TRAIN='../data/UNSW_NB15_testing-set.csv'   # 175k -> TRAIN (nama rilis tertukar)
UNSW_TEST ='../data/UNSW_NB15_training-set.csv'  #  82k -> TEST
OUTDIR='paper2_models'; os.makedirs(OUTDIR,exist_ok=True)
S3_BUCKET=os.environ.get('S3_BUCKET','ssh-detection-features-232032302717')
S3_PREFIX='unsw-far'; REGION=os.environ.get('AWS_REGION','ap-southeast-1')
SEED=42; H=0.01; EPS_TRAIN=0.1; ADV_RATIO=0.20; FEWSHOT_FRAC=0.01; MAXN=40000
print('files:', os.path.exists(CIC_PKL), os.path.exists(UNSW_TRAIN), os.path.exists(UNSW_TEST))
print('=== SEL 0 (setup) SELESAI ===')

In [ ]:
# --- SFM Model A (9 fitur) + util (konsisten 04/07/08) ---
MAP_A={'duration':('Flow Duration','dur'),'fwd_pkts':('Tot Fwd Pkts','spkts'),
       'bwd_pkts':('Tot Bwd Pkts','dpkts'),'fwd_bytes':('TotLen Fwd Pkts','sbytes'),
       'bwd_bytes':('TotLen Bwd Pkts','dbytes'),'fwd_mean':('Fwd Pkt Len Mean','smean'),
       'bwd_mean':('Bwd Pkt Len Mean','dmean'),'src_load':('Flow Byts/s','sload'),
       'dst_load':('Bwd Pkts/s','dload')}
CANON=list(MAP_A.keys()); IX={c:i for i,c in enumerate(CANON)}
def build_matrix(df, side):
    idx=0 if side=='cic' else 1
    cols=[MAP_A[c][idx] for c in CANON]
    out=df[cols].copy(); out.columns=CANON
    out=out.replace([np.inf,-np.inf],np.nan)
    out=out.fillna(out.median(numeric_only=True)).fillna(0.0)
    return out.astype(float).values
def make_xgb():
    return XGBClassifier(objective='binary:logistic',eval_metric='logloss',max_depth=8,
        learning_rate=0.1,n_estimators=200,subsample=0.8,colsample_bytree=0.8,
        n_jobs=-1,random_state=SEED,tree_method='hist')
def ev(yt,yp):
    return dict(mcc=float(matthews_corrcoef(yt,yp)),f1=float(f1_score(yt,yp,zero_division=0)),
                acc=float(accuracy_score(yt,yp)),confusion=confusion_matrix(yt,yp).tolist())
print('=== SEL 1 (SFM + util) SELESAI ===')

In [ ]:
# --- Serangan: finite-diff saliency + FGSM (untuk adversarial TRAINING) ---
def loss_bin(model,X,y):
    p=np.clip(model.predict_proba(X)[:,1],1e-15,1-1e-15); y=y.astype(float)
    return -(y*np.log(p)+(1-y)*np.log(1-p))
def saliency(model,X,y,h=H):
    n,m=X.shape; S=np.zeros((n,m))
    for i in range(m):
        Xp=X.copy(); Xp[:,i]+=h; Xm=X.copy(); Xm[:,i]-=h
        S[:,i]=(loss_bin(model,Xp,y)-loss_bin(model,Xm,y))/(2*h)
    return S
def fgsm(X,S,eps): return X+eps*np.sign(S)
def make_adv(model,Xs,ys,eps,cap=MAXN):
    n=min(cap,len(Xs)); idx=np.random.RandomState(SEED).choice(len(Xs),n,replace=False)
    Xsub,ysub=Xs[idx],ys[idx]; S=saliency(model,Xsub,ysub)
    return fgsm(Xsub,S,eps), ysub
print('=== SEL 2 (attack utils) SELESAI ===')

In [ ]:
# --- Muat CIC (un-scale) + label biner ---
with open(CIC_PKL,'rb') as f: d=pickle.load(f)
cic_feats=list(d['feature_names']); X=np.asarray(d['X'],float)
sc=d.get('scaler',None)
X_orig=X*sc.scale_+sc.mean_ if (sc is not None and hasattr(sc,'scale_')) else X
cic_df=pd.DataFrame(X_orig,columns=cic_feats)
benign=d.get('label_mapping',{}).get('Benign',0)
y_cic=(np.asarray(d['y'])!=benign).astype(int)
# --- Muat UNSW ---
unsw_tr=pd.read_csv(UNSW_TRAIN); unsw_te=pd.read_csv(UNSW_TEST)
y_utr=unsw_tr['label'].astype(int).values; y_ute=unsw_te['label'].astype(int).values
# --- Matriks + z-score PER DATASET (fit train saja) ---
Xc_all=build_matrix(cic_df,'cic')
Xc_tr_raw,Xc_te_raw,yc_tr,yc_te=train_test_split(Xc_all,y_cic,test_size=0.3,random_state=SEED,stratify=y_cic)
scc=StandardScaler().fit(Xc_tr_raw); Xc_tr=scc.transform(Xc_tr_raw); Xc_te=scc.transform(Xc_te_raw)
Xu_tr_raw=build_matrix(unsw_tr,'unsw'); Xu_te_raw=build_matrix(unsw_te,'unsw')
scu=StandardScaler().fit(Xu_tr_raw); Xu_tr=scu.transform(Xu_tr_raw); Xu_te=scu.transform(Xu_te_raw)
print('CIC tr/te:',Xc_tr.shape,Xc_te.shape,'| UNSW tr/te:',Xu_tr.shape,Xu_te.shape)
print('=== SEL 3 (muat + z-score) SELESAI ===')

In [ ]:
# --- Latih 4 varian utk satu arah (source -> target) ---
def build_robust_set(base_model, X_src_tr, y_src_tr):
    """D_robust = clean UNION adv(train) dgn rasio ADV_RATIO."""
    Xa,ya=make_adv(base_model,X_src_tr,y_src_tr,EPS_TRAIN)
    n_clean=len(X_src_tr); n_adv=min(int(n_clean*ADV_RATIO/(1-ADV_RATIO)),len(Xa))
    sel=np.random.RandomState(SEED).choice(len(Xa),n_adv,replace=False)
    Xr=np.vstack([X_src_tr,Xa[sel]]); yr=np.concatenate([y_src_tr,ya[sel]])
    return Xr,yr,n_adv

def train_variants(direction, X_src_tr,y_src_tr, X_tgt_tr,y_tgt_tr):
    print('='*60); print('ARAH:',direction); print('='*60)
    models={}
    # 1. baseline (source clean)
    m=make_xgb(); m.fit(X_src_tr,y_src_tr); models['baseline']=m
    # 2. fewshot: source + FEWSHOT_FRAC label target
    rng=np.random.RandomState(SEED); nfs=max(1,int(len(X_tgt_tr)*FEWSHOT_FRAC))
    ifs=rng.choice(len(X_tgt_tr),nfs,replace=False)
    Xfs=np.vstack([X_src_tr,X_tgt_tr[ifs]]); yfs=np.concatenate([y_src_tr,y_tgt_tr[ifs]])
    m=make_xgb(); m.fit(Xfs,yfs); models['fewshot']=m
    # 3. adv: source + adversarial training
    Xr,yr,nadv=build_robust_set(models['baseline'],X_src_tr,y_src_tr)
    m=make_xgb(); m.fit(Xr,yr); models['adv']=m
    # 4. fewshot_adv: (source + fewshot) lalu adv-training di atasnya (USULAN Paper 2)
    Xr2,yr2,nadv2=build_robust_set(models['fewshot'],Xfs,yfs)
    m=make_xgb(); m.fit(Xr2,yr2); models['fewshot_adv']=m
    print(f'  fewshot n_target={nfs} | adv n_adv={nadv} | fewshot_adv n_adv={nadv2}')
    return models, dict(n_fewshot=int(nfs), n_adv=int(nadv), n_adv_fs=int(nadv2))

models_c2u, meta_c2u = train_variants('CIC->UNSW', Xc_tr,yc_tr, Xu_tr,y_utr)
models_u2c, meta_u2c = train_variants('UNSW->CIC', Xu_tr,y_utr, Xc_tr,yc_tr)
print('=== SEL 4 (latih 4 varian x 2 arah) SELESAI ===')

In [ ]:
# --- Cek cepat: MCC clean in-domain & cross (belum evasion; itu di notebook 12) ---
def quick(direction, models, X_src_te,y_src_te, X_tgt_te,y_tgt_te):
    rows=[]
    for name,m in models.items():
        s=ev(y_src_te,m.predict(X_src_te))['mcc']
        t=ev(y_tgt_te,m.predict(X_tgt_te))['mcc']
        rows.append({'arah':direction,'model':name,'MCC_source_test':round(s,4),'MCC_target_test':round(t,4)})
    return rows
rows=[]
rows+=quick('CIC->UNSW',models_c2u,Xc_te,yc_te,Xu_te,y_ute)
rows+=quick('UNSW->CIC',models_u2c,Xu_te,y_ute,Xc_te,yc_te)
dfq=pd.DataFrame(rows); import IPython.display as ipd; ipd.display(dfq)
print('Catatan: MCC_target_test = generalisasi lintas-jaringan (clean). Evasion diuji di nb12.')
print('=== SEL 5 (cek cepat clean) SELESAI ===')

In [ ]:
# --- Simpan model + scaler + metadata, upload S3 ---
def save_dir(direction, models, scaler_src):
    dd=os.path.join(OUTDIR,direction.replace('->','_to_')); os.makedirs(dd,exist_ok=True)
    for name,m in models.items(): m.save_model(os.path.join(dd,f'{name}.json'))
    with open(os.path.join(dd,'scaler.pkl'),'wb') as f:
        pickle.dump({'mean':scaler_src.mean_,'scale':scaler_src.scale_,'features':CANON},f)
    return dd
save_dir('CIC->UNSW',models_c2u,scc)
save_dir('UNSW->CIC',models_u2c,scu)
meta=dict(generated=time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()),
          features=CANON, seed=SEED, eps_train=EPS_TRAIN, adv_ratio=ADV_RATIO,
          fewshot_frac=FEWSHOT_FRAC, variants=['baseline','fewshot','adv','fewshot_adv'],
          train_meta={'CIC->UNSW':meta_c2u,'UNSW->CIC':meta_u2c},
          quick_clean=rows)
json.dump(meta,open(os.path.join(OUTDIR,'paper2_pipeline_meta.json'),'w'),indent=2)
try:
    import boto3; s3=boto3.client('s3',region_name=REGION); up=0
    for root,_,files in os.walk(OUTDIR):
        for fn in files:
            lp=os.path.join(root,fn); rel=os.path.relpath(lp,OUTDIR).replace('\\\\','/')
            s3.upload_file(lp,S3_BUCKET,f'{S3_PREFIX}/paper2/{rel}'); up+=1
    print(f'upload {up} artefak -> s3://{S3_BUCKET}/{S3_PREFIX}/paper2/')
except Exception as e: print('upload gagal:',e)
print('=== SEL 6 (simpan + upload) SELESAI ===')
print('LANJUT: jalankan notebook 12 utk evaluasi evasion/adaptive pada model2 ini.')